---
# Chapter 8 — Is It Still True?

## Orientation

| Field | Value |
|-------|-------|
| Chapter | 8: Is It Still True? |
| Central question | How does the system track belief through time — distinguishing what was true then from what is true now? |
| Main concepts | Temporal trajectories, Supersession, Bitemporal state, Validity intervals, Late knowledge |
| Implementation | temporal_memory |
| Experiment | Temporal memory evaluation |
| Evidence status | Book result: core for temporal state, conditional otherwise |
| Depends on | Chapter 4 (derived graph), Chapter 2 (instrument) |

---

## What this notebook demonstrates

This chapter establishes that some memory questions depend on **trajectory, not merely the set of remembered events**. The notebook:

1. **Loads the temporal_memory implementation**
2. **Walks one claim through time** — queries historical truth and current truth separately
3. **Demonstrates supersession, effective time, and late knowledge** using the actual temporal representation
4. **Shows the frozen temporal run results** — value of ordered transitions and bitemporal state

> **Evidence status**: Book result. The frozen temporal run showed the value of ordered transitions and bitemporal state on historical and late-arrival cases while retaining recency for clean current-state queries.

## The chapter question

> **Is X still true?**

The same history must answer:

```text
What was true then?
What is true now?
What had been decided but not yet taken effect?
What did we know at that time?
```

This earns temporal state as a **real but scoped** requirement — needed when validity, ordering, effective time, late knowledge, correction or supersession can change the answer.

## Concepts in this chapter

In [ ]:
import sys
from pathlib import Path

def _find_repo_root(start):
    cur = Path(start).resolve()
    while True:
        if ((cur / "content").is_dir() and (cur / "notebooks").is_dir()
                and (cur / "solution").is_dir()):
            return cur
        if cur == cur.parent:
            raise RuntimeError("could not locate repository root")
        cur = cur.parent

REPO_ROOT = _find_repo_root(Path.cwd())
for _p in (str(REPO_ROOT), str(REPO_ROOT / "solution")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from notebooks.memory._support import load_chapter_metadata, render_table

meta = load_chapter_metadata(8)
concepts = (meta.get('chapter', {}).get('concepts')
            or meta.get('concepts', []))
render_table([
    {"Concept ID": c['id'], "Name": c['name'], "Status": c['status']}
    for c in concepts
], "Chapter 8 Concepts")

## Load the temporal_memory implementation

In [ ]:
from temporal_memory.fixtures import (
    E, canonical_migration, late_arrival, correction_case,
    future_effective, SUBJECT_BACKEND,
)
from temporal_memory.log import EventLog
from temporal_memory.query import TemporalEngine, ResolverCondition

print("Subject:", SUBJECT_BACKEND)
print("Resolver conditions:",
      [ResolverCondition.BAG, ResolverCondition.ARRIVAL,
       ResolverCondition.EVENT_TIME, ResolverCondition.ORDERED,
       ResolverCondition.TEMPORAL])

## The event-store timeline (canonical history)

Let's build the temporal units from the canonical history:

In [ ]:
# Freeze the canonical migration into an append-only event log.
events = canonical_migration()
log = EventLog()
for e in events:
    log.append(e, e.recorded_at)
print(f"Frozen log: {len(log.events())} events, gaps: {log.gaps()}")

render_table(
    [{"event": e.event_id, "type": e.event_type, "time": e.event_time,
      "value": e.value or "-", "effective": e.effective_from or "-",
      "evidence": ",".join(e.evidence_refs) or "-"}
     for e in events],
    "Canonical Migration Events (Ch8 Fixture)")

## Query the temporal memory at different times

In [ ]:
# Five resolver conditions on the same log. Current-state agrees
# everywhere here; the bitemporal spot checks are where they diverge.
for cond in (ResolverCondition.BAG, ResolverCondition.ARRIVAL,
             ResolverCondition.EVENT_TIME, ResolverCondition.ORDERED,
             ResolverCondition.TEMPORAL):
    a = TemporalEngine(log, cond).current(SUBJECT_BACKEND)
    print(f"{cond:14s} -> {a.value} [{a.status}] {a.detail}")

eng = TemporalEngine(log)
print("\nBitemporal spot checks (T3 temporal):")
for at in ("2026-04-01T00:00:00Z", "2026-06-20T00:00:00Z",
           "2026-08-01T00:00:00Z"):
    print(f"  valid at {at[:10]}: {eng.at(SUBJECT_BACKEND, at).value}")
print("  known at 2026-06-20:",
      eng.as_known(SUBJECT_BACKEND, "2026-06-20T00:00:00Z").value)
print("  transition history:")
for t in eng.history(SUBJECT_BACKEND):
    print(f"    {t['event']} [{t['type']}] {t['before']} -> {t['after']}")

## Supersession resolution

The temporal memory resolves supersession chains. Let's inspect the decisions:

In [ ]:
# Supersession is explicit lineage, not recency. The deploy event
# supersedes nothing by timestamp; the DECISION_MADE names its parents.
print(eng.explain_transition("ch8-decision"))
print()
print(eng.explain_transition("ch8-deploy"))

## Late knowledge / correction

What if we learn something later that changes our understanding of the past? The temporal memory handles late-arrival knowledge.

In [ ]:
# Late arrival, correction, and future-effective decisions each get
# their own log so the standpoint semantics stay visible.
def freeze_all(evts):
    lg = EventLog()
    for e in evts:
        lg.append(e, e.recorded_at)
    return lg

la_log = freeze_all(late_arrival())
print("late arrival, current:",
      TemporalEngine(la_log).current(SUBJECT_BACKEND).value)

c_log = freeze_all(correction_case())
c_eng = TemporalEngine(c_log)
print("correction log, current:", c_eng.current(SUBJECT_BACKEND).value)
print("correction history:")
for t in c_eng.history(SUBJECT_BACKEND):
    print(f"    {t['event']} [{t['type']}] {t['before']} -> {t['after']}")

f_log = freeze_all(future_effective())
f_eng = TemporalEngine(f_log)
decided = f_eng.as_known(SUBJECT_BACKEND, "2026-09-10T00:00:00Z")
print("future-effective, known at 2026-09-10:",
      decided.value, f"[{decided.status}]", decided.detail)

## The frozen temporal run results (from chapter)

> **Book result.** The frozen temporal run showed the value of ordered transitions and bitemporal state on historical and late-arrival cases while retaining recency for clean current-state queries.

In [ ]:
# Show the key findings
findings = [
    {"Query Class": "Clean current-state", "Temporal Machinery Needed": "No", "Result": "Recency works; simple retrieval sufficient"},
    {"Query Class": "Historical state (what was true then)", "Temporal Machinery Needed": "Yes", "Result": "Bitemporal state required; validity intervals essential"},
    {"Query Class": "Supersession (what replaced what)", "Temporal Machinery Needed": "Yes", "Result": "Ordered transitions + supersession chains required"},
    {"Query Class": "Late-arrival knowledge / correction", "Temporal Machinery Needed": "Yes", "Result": "Correction units with 'corrects' links; retroactive without rewriting history"},
    {"Query Class": "Effective time vs decision time", "Temporal Machinery Needed": "Yes", "Result": "Decision (Jul 11) ≠ production cutover (Jul 22); two valid_from dates"},
]
render_table(findings, "Temporal Query Classes and Machinery Requirements")

## What this establishes

- **Temporal state is real but scoped** — not needed for every query
- **Bitemporal state (valid_from, valid_until, supersedes) earns its place** on historical/late-arrival/supersession cases
- **Current-state questions can use recency** — temporal machinery unnecessary there
- **Late knowledge handled via correction units** — retroactive without rewriting history
- **Decision time ≠ effective time** — two different temporal dimensions

## What this does NOT establish

- No universal temporal machinery for all queries
- Real-corpus temporal extraction quality untested
- Multi-agent temporal disagreement untested

## Try it yourself

Add a contradictory unit and see how the temporal memory handles it. Query at different `as_of` dates to see the trajectory.

In [ ]:
# TRY IT YOURSELF: sweep the valid-time standpoint across the whole
# trajectory and watch the belief change exactly at the decision.
print("=== Valid-Time Trajectory for event-store.backend ===")
for d in ("2026-03-10", "2026-06-01", "2026-06-26", "2026-07-11",
          "2026-09-01"):
    a = eng.at(SUBJECT_BACKEND, d + "T00:00:00Z")
    print(f"  {d}: {a.value} [{a.status}]")

## Where this leads next

Chapter 9 asks: **What did we leave unfinished?** — tracking intentions and consequences as expected transitions.

> **See this chapter in code:** [Open the companion Jupyter notebook](memory 8-chapter.ipynb)